<a href="https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eng7ouda06/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This is a RANKING / SCORING task, not classification.

I'm not predicting a yes/no label for each page — I'm ordering all pages by
how much they under-capture clicks relative to what a page in their position
tier should get. The output is a ranked list, not a class.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target (proxy): CTR gap score = actual_ctr - expected_ctr_for_position_tier

This is a PROXY, not a directly observed outcome. "Expected CTR" is something
I define myself (mean CTR per position tier, computed from this same dataset),
not a fixed external ground truth. I'm treating it as a proxy for "is this
page under-capturing clicks for how well it ranks" — not a causal claim about
why.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: Precision@20 (or @50, depending on review capacity).

Of the top 20 pages my gap score flags for review, how many actually turn out
to be high-impression pages with a real, meaningful CTR shortfall when I
manually inspect them? Plain accuracy doesn't fit here — this is a ranked
queue a reviewer works top-down, not a classifier being graded on every row.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one content item (content_id), summarized over its trailing 90-day
window: impressions, clicks, ctr, avg_position, content_type, sessions,
engagement_rate.

In [ ]:
import pandas as pd

df = pd.read_csv("flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

cols = ["content_id", "impressions_90d", "clicks_90d", "ctr", "avg_position",
        "content_type", "sessions_90d", "engagement_rate"]
df[cols].head(10)

,content_id,impressions_90d,clicks_90d,ctr,avg_position,content_type,sessions_90d,engagement_rate
0,content_304f48230142,3803,29,0.76,10.6,keyword article,17,5.88
1,content_a1fb4e703a9e,15320,7,0.05,20.3,keyword article,9,0.00
2,content_9aa793d4d895,12581,11,0.09,36.5,keyword article,11,0.00
3,content_331d6c4de07b,11751,58,0.49,6.2,keyword article,78,1.28
4,content_d99b7a2d90ca,19140,24,0.13,44.0,keyword article,145,0.00
5,content_d4084a4bc775,3970,1,0.03,8.5,keyword article,5,0.00
6,content_9a34b442b552,20,0,0.00,7.0,keyword article,1,0.00
7,content_a63219c6e95a,1724,1,0.06,21.2,keyword article,28,3.57
8,content_5e6c160719bc,32574,29,0.09,46.0,keyword article,68,5.88
9,content_c27558df2b0c,1240,2,0.16,4.9,keyword article,3,0.00


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A flat rule like "CTR < 2% = bad" ignores that expected CTR varies hugely by
position (position 1 vs position 15 have completely different baseline click
rates) and by intent/content type. "Underperforming" only means something
relative to peers in the same position tier — that comparison has to be
computed from the data, it can't be hardcoded as a single threshold.

In [ ]:
df["position_tier"] = pd.cut(
    df["avg_position"],
    bins=[0, 3, 10, 20, 1000],
    labels=["1-3", "4-10", "11-20", "21+"]
)

df[df["avg_position"] > 0].groupby("position_tier")["ctr"].mean()

/tmp/ipykernel_2127/1559654637.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df[df["avg_position"] > 0].groupby("position_tier")["ctr"].mean()


,ctr
position_tier,
1-3,2.714303
4-10,0.651045
11-20,0.323443
21+,0.211333


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.